In [17]:
import pandas as pd
import json
import os
import random

In [18]:
bacteria_file = pd.read_json('..\\dissertation DLC content\\big_list_files\\names_report_nostrain.jsonl', lines=True, encoding='utf-8')['taxonomy']

yeasts_file = pd.read_json('..\\dissertation DLC content\\big_list_files\\names_report_yeast.jsonl', lines=True, encoding='utf-8')['taxonomy']

yeasts_file_2 = pd.read_json('..\\dissertation DLC content\\big_list_files\\basidio_names_report.jsonl', lines=True, encoding='utf-8')['taxonomy']

In [19]:
genus_file = pd.read_json('..\\dissertation DLC content\\big_list_files\\names_report_genus.jsonl', lines=True, encoding='utf-8')
genus_file_yeasts = pd.read_json('..\\dissertation DLC content\\big_list_files\\names_report_yeast_genera.jsonl', lines=True, encoding='utf-8')
genus_file_yeasts_2 = pd.read_json('list_files\\names_report_basidio_genera.jsonl', lines=True, encoding='utf-8')

genus_set = set()

genus_set.add('Mycoderma')
genus_set.add('Haloferax')
genus_set.add('Lachanchea')
genus_set.add('Enteroccocus')
genus_set.add('Dekkera')

def process_genera_file(file):
    for line in file:
        names = line.get('currentScientificName')
        heterotypic = names.get('heterotypicSynonyms', [{}])[0].get('name', '').replace('\"', '')
        homotypic = names.get('homotypicSynonyms', [{}])[0].get('name', '').replace('\"', '')
        genus_set.add(names.get('name'))
        if heterotypic != '':
            genus_set.add(heterotypic)
        if homotypic != '':
            genus_set.add(homotypic)
        
print(len(genus_set))

process_genera_file(genus_file['taxonomy'])
process_genera_file(genus_file_yeasts['taxonomy'])
process_genera_file(genus_file_yeasts_2['taxonomy'])

genus_list = list(genus_set)

print("shortest genus:", min([len(gen) for gen in genus_list]))

with open('list_files\\genus_names.json', 'w', encoding='utf-8') as f:
    json.dump(genus_list, f)

5
shortest genus: 3


In [20]:
genus_endings = set()

for genus in genus_list:
    genus_endings.add(genus[-3:])
    
print(len(genus_endings))
    
with open('list_files\\genus_endings.json', 'w', encoding='utf-8') as f:
    json.dump(list(genus_endings), f)

459


In [21]:

microbes_set = set()

skipping_names = ['rumen', 'anammox', 'nc10', 'anaplasma', 'wolbachia', 
                  'rickettsia', 'chlamydia', 'ehrlichia', 'coxiella', 'bartonella', 'legionella', 
                  'fecal', 'swine', 'francisella', 'cupriavidus', 'arcobacter', 'oral', 
                  'metal', 'gut', 'root', 'endosymbiont', 'gel', 'neorickettsia', 'rodentibacter', 
                  'sludge', 'intestinal', 'equine', 'oil', 'field', 'symbiont']

def get_microbes_from_file(file, limit):
    return_list = []
    i = 0
    for line in file:
        if i < limit:
            names = line.get('currentScientificName')
            heterotypic = names.get('heterotypicSynonyms', [{}])[0].get('name', '').replace('\"', '')
            homotypic = names.get('homotypicSynonyms', [{}])[0].get('name', '').replace('\"', '')
            basionym = names.get('basionym', {}).get('name', '').replace('\"', '')
            skipping_line = False
            for skip in skipping_names:
                combined = names.get('name').lower().split(' ') + heterotypic.lower().split(' ') + homotypic.lower().split(' ') + basionym.split(' ')
                if skip in combined:
                    skipping_line = True
                    break
            if skipping_line:
                continue
            microbes_set.add(names.get('name'))
            i += 1
            if heterotypic != '':
                microbes_set.add(heterotypic)
                i += 1
            if homotypic != '':
                microbes_set.add(homotypic)
                i += 1
            if basionym != '':
                microbes_set.add(basionym)
                i += 1
    return return_list

print(len(bacteria_file))

bacteria_list = get_microbes_from_file(bacteria_file, 50000)
random.shuffle(bacteria_list)

yeasts_list = get_microbes_from_file(yeasts_file, 10000)

yeasts_list_2 = get_microbes_from_file(yeasts_file_2, 10000)

microbes_list = bacteria_list + yeasts_list + yeasts_list_2

print(len(microbes_set))

530837
68973


In [22]:
numbers = ['0', '1', '2', '3', '4', '5', '6','7','8','9']

def get_numbers_in_candidate(candidate):
    nums = 0
    for l in list(candidate):
        if l in numbers:
            nums += 1
    return nums

species_list = []

microbes_patterns = []

species_3_endings = set()
species_4_endings = set()
species_5_endings = set()
species_6_endings = set()

species_5_beginnings = set()
species_6_beginnings = set()

#these are solely microbes in the form Genus species
microbes_genus_species = set()

for name in microbes_set:
    name_list = name.split(' ')
    genus = name_list[0]
    pattern = []
    candidate = None
    '''
    for name in name_list:
        subpattern = {'TEXT': name}
        if name.isnumeric():
            subpattern.update({'IS_DIGIT': True})
        if name.lower() == name:
            subpattern.update({'IS_LOWER': True})
        elif name.upper() == name:
            subpattern.update({'IS_UPPER': True})
        elif name.istitle():
            subpattern.update({'IS_TITLE': True})
        if name in genus_set:
            species_string = ' '.join(name_list[name_list.index(name)+1:])
            microbes_patterns.append({'label': 'MICROBE_NAME', 'pattern': [{'TEXT': f'{name[0]}', 'IS_SENT_START': True}, {'TEXT': '.', 'IS_PUNCT': True}, {'TEXT': species_string}], 'type': 'token'})
        pattern.append(subpattern)
    microbes_patterns.append({'label': 'MICROBE_NAME', 'pattern': pattern, 'type': 'token'})

    if name_list[0] == 'uncultured' or name_list[0] == 'unidentified':
        if 'subsp.' in name_list:
            if not candidate:
                candidate = name_list[name_list.index('subsp.')+1]
        elif 'sp.' in name_list:
            if name_list.index('sp.') != len(name_list) - 1:
                if not candidate:
                    candidate = name_list[name_list.index('sp.')+1]
        else:
            microbes_list.append({'label': 'MICROBE_NAME', 'pattern': [{'TEXT': name_list[0]}, {'TEXT': name_list[1], 'IS_TITLE': True}]})
                
        if name_list[1] in genus_set:
            if not candidate:
                candidate = ' '.join(name_list[2:])
        else:
            if not candidate:
                ' '.join(name_list[1:])
    '''
                
    #if its a species or subspecies
    if 'subsp.' in name_list:
        if not candidate:
            candidate = name_list[name_list.index('subsp.')-1]
            
    if 'sbsp.' in name_list:
        if not candidate:
            candidate = name_list[name_list.index('sbsp.')-1]

    if 'sp.' in name_list:
        #making sure 'sp.' doesnt appear at the end of the list before adding the species name that comes after it
        if name_list.index('sp.') != len(name_list) - 1:
            if not candidate:
                candidate = name_list[name_list.index('sp.')+1]
                
    if name_list[0] == 'Candidatus':
        if name_list[1] in genus_list:
            if not candidate:
                candidate = name_list[2]
        else:
            if not candidate:
                candidate = name_list[1]

    #otherwise, check if the genus appears, then grab the one after it
    if not candidate:
        for n in name_list:
            if n in genus_set:
                gen_ind = name_list.index(n)
                if gen_ind != len(name_list) - 1:
                    microbes_genus_species.add(str(name_list[gen_ind] + ' ' + name_list[gen_ind+1]))
                    if not candidate:
                        candidate = name_list[gen_ind+1:]
                        break
                    
    if candidate:
        if len(candidate) == 1:
            species = candidate[0]
            if not candidate == 'sp.' or candidate == 'subsp.':
                species_list.append(species)
                ending = species[-6:]
                if ending.isalpha() and ending.islower() and len(ending) > 2:
                    """ species_3_endings.add(species[-3:])
                    species_4_endings.add(species[-4:])
                    species_5_endings.add(species[-5:]) """
                    species_6_endings.add(ending)
                beginning = species[:6]
                if beginning.isalpha() and beginning.islower() and len(beginning) > 2:
                    #species_5_beginnings.add(beginning)
                    species_6_beginnings.add(beginning)
        
    '''       
    if not candidate:
        candidate = name
                
    nums_in_candidate = get_numbers_in_candidate(candidate)
    if len(candidate) > 4:
        #if less than half of the letters in the candidate are numbers
        if nums_in_candidate < len(candidate) // 2:
            species_list.append(candidate)
    '''
    
species_6_beginnings.add('oncorh')
species_6_endings.add('hynchi')
species_5_endings.add('ovara')
    
""" with open('list_files\\microbe_patterns.json', 'w', encoding='utf-8') as f:
    json.dump(microbes_patterns, f)
    
with open('list_files\\species_names.json', 'w', encoding='utf-8') as f:
    json.dump(species_list, f)
    
with open('list_files\\species_3_endings.json', 'w', encoding='utf-8') as f:
    json.dump(list(species_3_endings), f)
    
with open('list_files\\species_4_endings.json', 'w', encoding='utf-8') as f:
    json.dump(list(species_4_endings), f) """
    
with open('list_files\\species_6_endings.json', 'w', encoding='utf-8') as f:
    json.dump(list(species_6_endings), f)
    
with open('list_files\\species_6_beginnings.json', 'w', encoding='utf-8') as f:
    json.dump(list(species_6_beginnings), f)
    
with open('list_files\\microbes_genus_species.json', 'w', encoding='utf-8') as f:
    json.dump(list(microbes_genus_species), f)
    
'''
common patterns seen in microbes:
unidentified/uncultured genus species
unidentified/uncultured species
genus species
genus species subsp. subspecies
genus sp. species
G. species
'''

new_vocab = species_list + genus_list

#with open('list_files\\new_microbial_vocab.json', 'w', encoding='utf-8')

In [23]:
num = 0

In [24]:
print("all species:", len(species_list))
print("all Genus species:", len(microbes_genus_species))

print(list(microbes_genus_species)[:100])

all species: 24888
all Genus species: 25174
['Actinomyces roseoviolaceus', 'Agaricus albus', 'Desulforamulus ruminis', 'Mycena rutilantiformis', 'Ascosphaera apis', 'Spirillum minutulum', 'Cronartium comptoniae', 'Elaphomyces virgatosporus', 'Schenella simplex', 'Cystofilobasidium feraegula', 'Agaricus laccatus', 'Acholeplasma entomophilum', 'Coprinus niveus', 'Hypochnicium punctulatum', 'Phyllactinia roboris', 'Tepidibacter formicigenes', 'Lecidea scopulicola', 'Actinomyces violascens', 'Cortinarius vitiosus', 'Brevibacillus invocatus', 'Spiromastigoides alatospora', 'Marinomonas basaltis', 'Helicobasidium purpureum', 'Thermomicrobium roseum', 'Cryocola antiquus', 'Balansia gaduae', 'Entyloma deliliae', 'Micrococcus epidermidis', 'Paenibacillus thermophilus', 'Actinomyces gobitricini', 'Pertusaria flavicunda', 'Cylindrocladium penicilloides', 'Synechococcus kpj425f', 'Arthrobotrys gephyropaga', 'Selenomonas flueggei', 'Bradyrhizobium pachyrhizi', 'Ramularia endophylla', 'Nocardiopsis 